<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/01_introduction/introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction: Meet Helcon and the Data

This is the first session of the course. Over the sessions we build one running system, the Helcon Support Copilot, piece by piece. Helcon is a fictional SaaS helpdesk whose support team is overwhelmed by customer tickets, and the copilot is the AI that reads each ticket, routes it to the right queue, answers it from the knowledge base, and knows when it does not know. This notebook is a tour of the raw material: the tickets, the knowledge base, and a set of hard cases that the whole course is built on.

Every session adds a real piece, and they all load the same data the same way through the shared SDK (system_io).

| Session | What we build | Built from |
|---|---|---|
| session 3 | A rule-based ticket reader that breaks on real language | the tickets |
| session 4 | Semantic search over the tickets | the tickets |
| session 5 | The intuition for why models hallucinate | the tickets |
| session 6 | The triage classifier that routes tickets | the tickets and their labels |
| session 7 | The retriever that answers from the knowledge base | the knowledge base |
| session 8 | The agent that wires triage and retrieval together | everything |
| session 9 | The whole copilot, end to end | everything |


## Step 1: Set up the shared data

Run this cell first. It pulls in the shared Helcon data and the small helper, the shared SDK (system_io), that every notebook in the course uses to load it. On Colab it clones the course repo. Locally the repo is already on disk.


In [ ]:
REPO = "https://github.com/acastellanos-ie/NLP-MBDS-EN-NEW"
import os, sys
if not os.path.exists("NLP-MBDS-EN") and not os.path.exists("../system/system_io.py"):
    !git clone --depth 1 -q $REPO NLP-MBDS-EN
for cand in ("../system", "NLP-MBDS-EN/system", "system"):
    if os.path.exists(os.path.join(cand, "system_io.py")):
        sys.path.insert(0, cand); break

import pandas as pd
import matplotlib.pyplot as plt
from system_io import load_seed, system_status

system_status()

## Step 2: The tickets, Helcon's raw material

Every message a customer sends Helcon's support team is one row. This is the text we will parse in session 3, search in session 4, and learn to classify in session 6. Let's load it and read a few.


In [ ]:
# Load the tickets table through the shared SDK and report its shape.
tickets = pd.DataFrame(load_seed("tickets"))
print(f"{len(tickets)} support tickets, columns: {list(tickets.columns)}\n")

# Print a fixed random sample so the tour is reproducible: category then text.
for _, row in tickets.sample(6, random_state=0).iterrows():
    print(f"[{row['category']:<13}] {row['text']}")

## Step 3: What are customers writing about?

Each ticket is labelled with one of 10 categories, the queues Helcon's team routes to. This label is what the triage classifier in session 6 learns to predict. Notice the mix is uneven, the way a real support inbox is: billing and account problems dominate, while feature requests and outages are rarer.


In [ ]:
# Count how many tickets fall in each category (queue).
counts = tickets["category"].value_counts()

# Horizontal bar chart, reversed so the largest queue sits on top.
plt.figure(figsize=(9, 5))
plt.barh(counts.index[::-1], counts.values[::-1], color="#4C72B0")
plt.xlabel("number of tickets")
plt.title("Helcon support tickets by category")
for i, v in enumerate(counts.values[::-1]):
    plt.text(v + 2, i, str(v), va="center")  # label each bar with its count
plt.tight_layout()
plt.show()

## Step 4: How long are the tickets?

Support messages are short, a sentence or two. That matters, because it shapes how we chunk, embed, and feed them to the model later in the course.


In [ ]:
# Measure each ticket's length in words (whitespace split).
lengths = tickets["text"].str.split().str.len()

# Histogram of word counts to show how short the tickets are.
plt.figure(figsize=(9, 4.5))
plt.hist(lengths, bins=range(0, lengths.max() + 3), color="#55A868", edgecolor="white")
plt.xlabel("words per ticket")
plt.ylabel("number of tickets")
plt.title("Most tickets are one or two sentences")
plt.tight_layout()
plt.show()

print(f"median length: {int(lengths.median())} words   |   "
      f"shortest: {lengths.min()}   longest: {lengths.max()}")

## Step 5: Each queue speaks its own vocabulary

Why can a model tell a billing ticket from an account ticket? Because the words differ. Below are the most frequent (non-stopword) terms in each of the six largest queues, the lexical fingerprint the classic baselines in session 6 will lean on.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Take the six largest queues and show their most frequent words side by side.
top_cats = tickets["category"].value_counts().index[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, cat in zip(axes.ravel(), top_cats):
    texts = tickets.loc[tickets["category"] == cat, "text"]
    # Count word frequencies for this queue, dropping common English stopwords.
    cv = CountVectorizer(stop_words="english", max_features=2000)
    X = cv.fit_transform(texts)
    freqs = X.sum(axis=0).A1               # total count per word across the queue
    vocab = cv.get_feature_names_out()
    # Keep the 7 most frequent words, reversed so the top word plots at the top.
    top = sorted(zip(vocab, freqs), key=lambda p: -p[1])[:7][::-1]
    ax.barh([w for w, _ in top], [v for _, v in top], color="#C44E52")
    ax.set_title(cat)
fig.suptitle("The most common words in each support queue", fontsize=14)
plt.tight_layout()
plt.show()

Pick a queue and read a handful of its real tickets. Change `CATEGORY` and re-run.


In [ ]:
# Default to REFUND if the dataset has it, otherwise fall back to BILLING.
CATEGORY = "REFUND" if "REFUND" in tickets["category"].values else "BILLING"
# try: ACCOUNT, INTEGRATIONS, CANCELLATION, OUTAGE, DATA_PRIVACY, FEATURE_REQUEST

# Pull a small fixed sample of that queue and print the raw ticket text.
sample = tickets[tickets["category"] == CATEGORY].sample(5, random_state=1)
print(f"Five real '{CATEGORY}' tickets:\n")
for _, row in sample.iterrows():
    print(f"  - {row['text']}")

## Step 6: The knowledge base, what the copilot answers from

Tickets are the questions. Helcon's knowledge base (its help center and policies) is where the answers live. These articles and policy documents are the corpus the retriever in session 7 and the final copilot in session 9 ground their answers in. One policy is deliberately held back, a stale draft we use much later to show why grounded does not always mean correct.


In [ ]:
# Load the two knowledge-base sources: help-center articles and policy documents.
kb = load_seed("kb")
policies = load_seed("policies")
print(f"{len(kb)} help-center articles + {len(policies)} policy documents\n")

# List the title and category of every document so we can see the corpus.
print("Help center:")
for d in kb:
    print(f"  - {d['title']}  ({d['category']})")
print("\nPolicies:")
for d in policies:
    print(f"  - {d['title']}  ({d['category']})")

## Step 7: A preview of trouble, the hard cases

Not every ticket is easy. The dataset includes curated hard cases, real-sounding messages engineered to break naive systems. In session 3 we will watch a classical grammar-based reader fall for exactly these. A taste:


In [ ]:
# Load the curated hard cases, grouped by the linguistic trap they illustrate.
probes = load_seed("probes")
labels = {
    "passive_voice": "Passive voice (who did what?)",
    "negation":      "Negation (the opposite of what it says)",
    "synonym":       "Synonyms (no shared keywords)",
    "multi_category":"Two problems in one ticket",
}
# Show one example per trap: the text and what it is meant to break.
for group, title in labels.items():
    item = probes[group][0]
    print(f"{title}")
    print(f'   "{item["text"]}"')
    print(f"   -> {item['shows']}\n")

## Takeaway

That is the whole raw material of the course in one place: 748 short support tickets labelled into 10 queues, a knowledge base of help-center articles and policies, and a set of hard cases designed to break our first attempts. Everything we build (the parser, the retriever, the triage classifier, the agent, the final copilot) is built on exactly this data, loaded the same way with `load_seed(...)` in every session.

Next session we meet the first technique and the first failure. See you in session 3.

Sneak peek: in session 4 we will turn these 748 tickets into points on a map and watch them cluster by topic, with no labels at all.
